# 06 - Equity / context data: Census, asthma, health survey
**Input:** `data/raw/census_acs5_2022_zcta.csv` (needs a free Census API key, see `scripts/fetch_data.py`),
`notebooks/data/NYC_asthma_nta_rates_2017_2019.csv`, `notebooks/data/NYC_community_health_survey.csv`
**Output:** `data/interim/census_zone.csv`, `data/interim/asthma_nta.csv`, `data/interim/asthma_borough.csv`, `data/interim/health_survey_citywide.csv`

These do **not** drive the forecaster. They (1) weight demand by population, (2) audit fairness - whether slow
response clusters in low-income / low-vehicle-access areas - and (3) provide the public-health narrative.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
RAW, INTERIM, PROCESSED = ROOT / "data/raw", ROOT / "data/interim", ROOT / "data/processed"
NB_DATA = ROOT / "notebooks/data"
INTERIM.mkdir(parents=True, exist_ok=True); PROCESSED.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

## Census ACS (population, income, vehicle access, poverty)

In [2]:
cp = RAW / "census_acs5_2022_zcta.csv"
if cp.exists():
    c = pd.read_csv(cp, dtype={"zcta": str})
    c = c.replace(-666666666, np.nan)                     # Census sentinel for 'not available'
    xw = pd.read_csv(INTERIM / "zip_to_modzcta.csv", dtype=str)
    c = c.merge(xw, left_on="zcta", right_on="zipcode", how="inner")
    c["no_vehicle"] = c.owner_no_vehicle + c.renter_no_vehicle
    agg = c.groupby("zone").agg(population=("population", "sum"), occupied_units=("occupied_housing_units", "sum"),
                               no_vehicle=("no_vehicle", "sum"), poverty_universe=("poverty_universe", "sum"),
                               below_poverty=("below_poverty", "sum"),
                               median_income=("median_household_income", "median"), median_age=("median_age", "median"))
    agg["pct_no_vehicle"] = agg.no_vehicle / agg.occupied_units
    agg["pct_poverty"] = agg.below_poverty / agg.poverty_universe
    agg.reset_index().to_csv(INTERIM / "census_zone.csv", index=False)
    print(len(agg), "zones with census data"); display(agg.describe().round(2))
else:
    print("SKIPPED: census file missing. Get a free key at https://api.census.gov/data/key_signup.html and run\n"
          "  CENSUS_API_KEY=<key> python scripts/fetch_data.py census")

SKIPPED: census file missing. Get a free key at https://api.census.gov/data/key_signup.html and run
  CENSUS_API_KEY=<key> python scripts/fetch_data.py census


## Asthma rates by neighbourhood (NTA)

In [3]:
a = pd.read_csv(NB_DATA / "NYC_asthma_nta_rates_2017_2019.csv")
a.columns = ["period", "geo_type", "geo_desc", "geo_id", "geo_rank", "boro_id", "borough", "nta", "area", "avg_annual_n", "rate_per_10k", "age_adj_rate_per_10k"]
print(a.geo_type.value_counts().to_dict())
for c in ["avg_annual_n", "rate_per_10k", "age_adj_rate_per_10k"]:
    a[c] = pd.to_numeric(a[c], errors="coerce")
print("NTAs with missing rate or count (dropped):", int(a[a.geo_type == "NTA"][["rate_per_10k", "avg_annual_n"]].isna().any(axis=1).sum()))
nta = a[a.geo_type == "NTA"].dropna(subset=["rate_per_10k", "avg_annual_n"]).copy()
nta["borough"] = nta.borough.str.upper()
nta[["borough", "nta", "avg_annual_n", "rate_per_10k", "age_adj_rate_per_10k"]].to_csv(INTERIM / "asthma_nta.csv", index=False)
boro = nta.groupby("borough").apply(lambda g: pd.Series({"avg_annual_n": g.avg_annual_n.sum(),
        "rate_per_10k_weighted": np.average(g.rate_per_10k, weights=g.avg_annual_n)}), include_groups=False)
boro.to_csv(INTERIM / "asthma_borough.csv"); print(boro.round(1))

{'NTA': 188}
NTAs with missing rate or count (dropped): 8
               avg_annual_n  rate_per_10k_weighted
borough                                           
BRONX               15584.0                  190.7
BROOKLYN            13317.0                  115.2
MANHATTAN            6604.0                   89.5
QUEENS               7945.0                   64.1
STATEN ISLAND        2111.0                   96.3


## Community health survey (citywide yearly prevalence)

In [4]:
s = pd.read_csv(NB_DATA / "NYC_community_health_survey.csv")
s = s.rename(columns={"Prevelance": "row"})
s = s[s.row.str.startswith("Prevalence")].drop(columns="row")     # drop the CI rows
s["Year"] = s.Year.astype(int)
for c in s.columns[1:]: s[c] = pd.to_numeric(s[c], errors="coerce")
s.columns = [c.lower().replace(" ", "_").replace(",", "").replace("(", "").replace(")", "").replace("+", "plus") for c in s.columns]
print(s.shape); s.to_csv(INTERIM / "health_survey_citywide.csv", index=False); s.head()

(11, 11)


,year,no_health_insurance,did_not_get_needed_medical_care,no_personal_doctor,drinks_1_or_more_sugar-sweetened_beverages_per_day,smoking_status_current_smokers,binge_drinking,obesity,colon_cancer_screening_adults_age_50plus_colonoscopy,self-reported_health_status_excellent/very_good/good,flu_shot_in_last_12_months_adults_ages_65plus_not_age-adjusted
0,2020,12.6,12.4,28.8,22.3,10.9,16.6,25.4,71.8,80.8,69.8
3,2019,12.7,12.4,16.2,22.1,11.9,18.9,24.8,70.7,76.6,67.6
6,2018,11.6,11.1,15.6,23.6,12.8,17.8,25.7,69.1,76.8,62.8
9,2017,11.8,10.3,15.2,23.0,13.4,17.3,25.1,69.9,76.8,66.1
12,2016,10.9,10.5,16.8,22.7,13.1,17.0,23.6,68.5,78.0,65.4
